# Phase 10: Learned Discrete OSP Candidate Selector

Goal: learn over the exact discrete OSP candidate family instead of a continuous surrogate.

This notebook keeps the final filter exact:
- generate the valid `(a,b)` candidates from the OSP search space
- learn a small selector over those candidates
- jointly train the selector with a lightweight reconstructor
- anneal and briefly reheat the selector temperature to avoid poor local minima


# Learnable MSFA Research Track

This notebook is part of the 10-day publication-oriented extension track.

## Run discipline
- Keep one experiment purpose per notebook.
- Save metrics and checkpoints to the configured project root.
- Do not silently change data splits or loss definitions across notebooks.
- When a result becomes final, export both numeric artifacts and a figure/table asset.


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted.")
except Exception as exc:
    print("Drive mount skipped:", exc)


In [ ]:
import csv
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path("/content/drive/MyDrive/Msa-Osp")
PATCH_PATH = PROJECT_ROOT / "dataset_patches.npz"
BASELINE_CKPT_PATH = PROJECT_ROOT / "unet_baseline_best.pth"
HISTORY_PATH = PROJECT_ROOT / "learned_osp_selector_history.csv"
CKPT_PATH = PROJECT_ROOT / "learned_osp_selector_best.pth"
SWEEP_HISTORY_PATH = PROJECT_ROOT / "learned_osp_selector_hard_sweep.csv"
REFINE_HISTORY_PATH = PROJECT_ROOT / "learned_osp_selector_refine_history.csv"
REFINE_CKPT_PATH = PROJECT_ROOT / "learned_osp_selector_refine_best.pth"
FIG_PATH = PROJECT_ROOT / "learned_osp_selector_pattern.png"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8
SELECTOR_EPOCHS = 40
HARD_SWEEP_EPOCHS = 8
FINAL_REFINE_EPOCHS = 16
MODEL_LR = 1e-4
SELECTOR_LR = 2e-2
REFINE_LR = 5e-5
BAND_COUNT = 16
TILE_SIZE = 16
BASE = 32
TEMP_START = 2.5
TEMP_END = 0.25
REHEAT_START = 24
REHEAT_LENGTH = 6
REHEAT_TEMP = 1.20
LAMBDA_SCORE = 5e-2
LAMBDA_SELECTOR_ENT = 2e-3
TRAIN_SAM_WEIGHT = 5e-2
USE_GUMBEL = True

print("Device:", DEVICE)
print("Warm-start baseline exists:", BASELINE_CKPT_PATH.exists())
print("Notebook goal: exact OSP candidate selection, then hard-filter fine-tuning.")


In [ ]:
data = np.load(PATCH_PATH)
train_target = data["train"]
val_target = data["val"]

class CubeDataset(Dataset):
    def __init__(self, cubes):
        self.cubes = cubes

    def __len__(self):
        return len(self.cubes)

    def __getitem__(self, idx):
        return torch.from_numpy(self.cubes[idx]).permute(2, 0, 1).float()

train_loader = DataLoader(CubeDataset(train_target), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(CubeDataset(val_target), batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class UNet2D(nn.Module):
    def __init__(self, in_ch=16, out_ch=16, base=BASE):
        super().__init__()
        self.enc1 = DoubleConv(in_ch, base)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(base, base * 2)
        self.pool2 = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base * 2, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = DoubleConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = DoubleConv(base * 2, base)
        self.final = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        b = self.bottleneck(self.pool2(e2))
        d2 = self.dec2(torch.cat([self.up2(b), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.final(d1)

def compute_psnr(pred, target, eps=1e-8):
    mse = torch.mean((pred - target) ** 2)
    return 20 * torch.log10(1.0 / torch.sqrt(mse + eps))

def spectral_angle_mapper(pred, target, eps=1e-8):
    dot = torch.sum(pred * target, dim=1)
    pred_norm = torch.norm(pred, dim=1)
    target_norm = torch.norm(target, dim=1)
    cos_theta = torch.clamp(dot / (pred_norm * target_norm + eps), -1 + eps, 1 - eps)
    return torch.rad2deg(torch.acos(cos_theta)).mean()

def spectral_angle_loss(pred, target, eps=1e-8):
    dot = torch.sum(pred * target, dim=1)
    pred_norm = torch.norm(pred, dim=1)
    target_norm = torch.norm(target, dim=1)
    cos_theta = torch.clamp(dot / (pred_norm * target_norm + eps), -1 + eps, 1 - eps)
    return torch.acos(cos_theta).mean()

def make_candidate_tile(a, b):
    idx = torch.arange(1, BAND_COUNT + 1, dtype=torch.int64)
    I, J = torch.meshgrid(idx, idx, indexing="ij")
    return torch.remainder(I * a + J * b, BAND_COUNT) + 1

def is_uniform_candidate(tile):
    target = set(range(1, BAND_COUNT + 1))
    row_ok = all(set(tile[r, :].tolist()) == target for r in range(TILE_SIZE))
    col_ok = all(set(tile[:, c].tolist()) == target for c in range(TILE_SIZE))
    return row_ok and col_ok

def candidate_score(tile):
    idx = torch.arange(1, TILE_SIZE + 1, dtype=torch.float32)
    I, J = torch.meshgrid(idx, idx, indexing="ij")
    points = torch.stack([I.reshape(-1), J.reshape(-1), tile.float().reshape(-1)], dim=1)
    return torch.pdist(points).min().item()

def tile_to_mask(tile):
    return F.one_hot((tile - 1).long(), num_classes=BAND_COUNT).permute(2, 0, 1).float()

def apply_candidate_mask(x, mask):
    _, _, h, w = x.shape
    tile_full = mask.repeat(1, h // TILE_SIZE, w // TILE_SIZE)
    return x * tile_full.unsqueeze(0)

candidate_records = []
for a in range(1, BAND_COUNT // 2 + 1):
    for b in range(a, BAND_COUNT // 2 + 1):
        if math.gcd(a, BAND_COUNT) != 1 or math.gcd(b, BAND_COUNT) != 1:
            continue
        tile = make_candidate_tile(a, b)
        if len(torch.unique(tile)) != BAND_COUNT:
            continue
        if not is_uniform_candidate(tile):
            continue
        candidate_records.append(
            {
                "a": a,
                "b": b,
                "tile": tile,
                "score_raw": candidate_score(tile),
            }
        )

candidate_tiles = torch.stack([tile_to_mask(rec["tile"]) for rec in candidate_records], dim=0)
candidate_scores_raw = torch.tensor([rec["score_raw"] for rec in candidate_records], dtype=torch.float32)
candidate_scores_norm = (candidate_scores_raw - candidate_scores_raw.min()) / (
    candidate_scores_raw.max() - candidate_scores_raw.min() + 1e-8
)
candidate_ab = torch.tensor([[rec["a"], rec["b"]] for rec in candidate_records], dtype=torch.int64)

print("Discrete OSP candidate count:", len(candidate_records))
print("Candidate (a,b) pairs:", [tuple(map(int, ab)) for ab in candidate_ab.tolist()])
print("Candidate raw scores:", candidate_scores_raw.tolist())

class OSPCandidateSelector(nn.Module):
    def __init__(self, candidate_tiles, candidate_scores_raw, candidate_scores_norm, candidate_ab):
        super().__init__()
        self.register_buffer("candidate_tiles", candidate_tiles)
        self.register_buffer("candidate_scores_raw", candidate_scores_raw)
        self.register_buffer("candidate_scores_norm", candidate_scores_norm)
        self.register_buffer("candidate_ab", candidate_ab)
        self.logits = nn.Parameter(0.25 * candidate_scores_norm.clone())

    def probs(self, temp, training=True):
        if training and USE_GUMBEL:
            return F.gumbel_softmax(self.logits, tau=temp, hard=False, dim=0)
        return torch.softmax(self.logits / temp, dim=0)

    def soft_tile(self, temp, training=True):
        probs = self.probs(temp, training=training)
        return torch.sum(probs.view(-1, 1, 1, 1) * self.candidate_tiles, dim=0)

    def expected_score(self, temp, training=True):
        probs = self.probs(temp, training=training)
        return torch.sum(probs * self.candidate_scores_norm)

    def selector_entropy(self, temp, training=True, eps=1e-8):
        probs = self.probs(temp, training=training)
        return -(probs * torch.log(probs + eps)).sum()

    def hard_index(self):
        return int(torch.argmax(self.logits).item())

    def hard_tile(self):
        idx = self.hard_index()
        return self.candidate_tiles[idx].argmax(dim=0) + 1

    def selected_ab(self):
        idx = self.hard_index()
        ab = self.candidate_ab[idx].tolist()
        return idx, (int(ab[0]), int(ab[1]))

    def selected_raw_score(self):
        return float(self.candidate_scores_raw[self.hard_index()].item())

    def forward(self, x, temp, training=True):
        _, _, h, w = x.shape
        tile = self.soft_tile(temp=temp, training=training)
        tile_full = tile.repeat(1, h // TILE_SIZE, w // TILE_SIZE)
        return x * tile_full.unsqueeze(0)


In [ ]:
def selector_temperature(epoch):
    base = TEMP_START + (TEMP_END - TEMP_START) * (epoch - 1) / max(SELECTOR_EPOCHS - 1, 1)
    if REHEAT_START <= epoch < REHEAT_START + REHEAT_LENGTH:
        frac = (epoch - REHEAT_START) / max(REHEAT_LENGTH - 1, 1)
        return REHEAT_TEMP + frac * (base - REHEAT_TEMP)
    return base

model = UNet2D().to(DEVICE)
selector = OSPCandidateSelector(
    candidate_tiles=candidate_tiles,
    candidate_scores_raw=candidate_scores_raw,
    candidate_scores_norm=candidate_scores_norm,
    candidate_ab=candidate_ab,
).to(DEVICE)

if BASELINE_CKPT_PATH.exists():
    checkpoint = torch.load(BASELINE_CKPT_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    print("Warm-started reconstructor from fixed-MSFA baseline.")
else:
    print("Starting reconstructor from scratch.")

optimizer = torch.optim.Adam(
    [
        {"params": model.parameters(), "lr": MODEL_LR},
        {"params": selector.parameters(), "lr": SELECTOR_LR},
    ]
)
criterion = nn.L1Loss()
best_psnr = -float("inf")
history = []

for epoch in range(1, SELECTOR_EPOCHS + 1):
    temp = selector_temperature(epoch)
    model.train()
    selector.train()
    train_loss = 0.0
    train_psnr = 0.0

    for x in train_loader:
        x = x.to(DEVICE)
        sensed = selector(x, temp=temp, training=True)
        pred = model(sensed)
        recon_loss = criterion(pred, x)
        score_loss = -selector.expected_score(temp=temp, training=True)
        selector_ent = selector.selector_entropy(temp=temp, training=True)
        loss = recon_loss + LAMBDA_SCORE * score_loss + LAMBDA_SELECTOR_ENT * selector_ent
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_psnr += compute_psnr(pred, x).item()

    model.eval()
    selector.eval()
    val_psnr = 0.0
    val_sam = 0.0
    with torch.no_grad():
        for x in val_loader:
            x = x.to(DEVICE)
            pred = model(selector(x, temp=temp, training=False))
            val_psnr += compute_psnr(pred, x).item()
            val_sam += spectral_angle_mapper(pred, x).item()

    train_loss /= len(train_loader)
    train_psnr /= len(train_loader)
    val_psnr /= len(val_loader)
    val_sam /= len(val_loader)
    probs = selector.probs(temp=temp, training=False).detach().cpu()
    selected_idx, (selected_a, selected_b) = selector.selected_ab()
    selected_score_raw = selector.selected_raw_score()
    expected_score_norm = selector.expected_score(temp=temp, training=False).item()
    selector_entropy = selector.selector_entropy(temp=temp, training=False).item()
    top_prob = probs.max().item()
    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_psnr": train_psnr,
            "val_psnr": val_psnr,
            "val_sam_deg": val_sam,
            "temperature": temp,
            "selected_index": selected_idx,
            "selected_a": selected_a,
            "selected_b": selected_b,
            "selected_score_raw": selected_score_raw,
            "expected_score_norm": expected_score_norm,
            "selector_entropy": selector_entropy,
            "top_probability": top_prob,
        }
    )
    print(
        f"Epoch {epoch:02d} | train PSNR {train_psnr:.2f} dB | "
        f"val PSNR {val_psnr:.2f} dB | val SAM {val_sam:.2f} deg | "
        f"temp {temp:.2f} | sel ({selected_a},{selected_b}) | "
        f"score {selected_score_raw:.3f} | top p {top_prob:.3f}"
    )

    if val_psnr > best_psnr:
        best_psnr = val_psnr
        CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)
        torch.save(
            {
                "epoch": epoch,
                "model": model.state_dict(),
                "selector": selector.state_dict(),
                "best_val_psnr": best_psnr,
                "temperature": temp,
                "selected_index": selected_idx,
                "selected_a": selected_a,
                "selected_b": selected_b,
                "selected_score_raw": selected_score_raw,
                "candidate_ab": candidate_ab.cpu().numpy(),
                "candidate_scores_raw": candidate_scores_raw.cpu().numpy(),
                "candidate_probs": probs.numpy(),
                "hard_tile": selector.hard_tile().detach().cpu().numpy(),
            },
            CKPT_PATH,
        )

HISTORY_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(HISTORY_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
    writer.writeheader()
    writer.writerows(history)

# Stage 2: sweep over exact geometry-optimal OSP candidates under hard training, then fine-tune the winner.
stage1_checkpoint = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
selector.load_state_dict(stage1_checkpoint["selector"])
selector.eval()
stage1_probs = torch.tensor(stage1_checkpoint["candidate_probs"], dtype=torch.float32)
geometry_best = candidate_scores_raw.max()
geometry_indices = torch.where(torch.abs(candidate_scores_raw - geometry_best) < 1e-6)[0].tolist()
geometry_indices = sorted(geometry_indices, key=lambda i: float(stage1_probs[i]), reverse=True)

sweep_rows = []
best_candidate_idx = None
best_candidate_state = None
best_candidate_psnr = -float("inf")
best_candidate_sam = float("inf")

for cand_idx in geometry_indices:
    cand_a = int(candidate_ab[cand_idx, 0].item())
    cand_b = int(candidate_ab[cand_idx, 1].item())
    cand_mask = candidate_tiles[cand_idx].to(DEVICE)
    cand_model = UNet2D().to(DEVICE)
    cand_model.load_state_dict(stage1_checkpoint["model"])
    cand_optimizer = torch.optim.Adam(cand_model.parameters(), lr=REFINE_LR)
    cand_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(cand_optimizer, T_max=HARD_SWEEP_EPOCHS)
    cand_best_psnr = -float("inf")
    cand_best_sam = float("inf")
    cand_best_state = None

    for sweep_epoch in range(1, HARD_SWEEP_EPOCHS + 1):
        cand_model.train()
        for x in train_loader:
            x = x.to(DEVICE)
            sensed = apply_candidate_mask(x, cand_mask)
            pred = cand_model(sensed)
            loss = criterion(pred, x) + TRAIN_SAM_WEIGHT * spectral_angle_loss(pred, x)
            cand_optimizer.zero_grad()
            loss.backward()
            cand_optimizer.step()

        cand_scheduler.step()
        cand_model.eval()
        cand_val_psnr = 0.0
        cand_val_sam = 0.0
        with torch.no_grad():
            for x in val_loader:
                x = x.to(DEVICE)
                pred = cand_model(apply_candidate_mask(x, cand_mask))
                cand_val_psnr += compute_psnr(pred, x).item()
                cand_val_sam += spectral_angle_mapper(pred, x).item()

        cand_val_psnr /= len(val_loader)
        cand_val_sam /= len(val_loader)
        if (cand_val_psnr > cand_best_psnr) or (
            abs(cand_val_psnr - cand_best_psnr) < 1e-6 and cand_val_sam < cand_best_sam
        ):
            cand_best_psnr = cand_val_psnr
            cand_best_sam = cand_val_sam
            cand_best_state = {k: v.detach().cpu().clone() for k, v in cand_model.state_dict().items()}

    sweep_rows.append(
        {
            "candidate_index": cand_idx,
            "a": cand_a,
            "b": cand_b,
            "score_raw": float(candidate_scores_raw[cand_idx].item()),
            "selector_probability": float(stage1_probs[cand_idx].item()),
            "best_val_psnr": cand_best_psnr,
            "best_val_sam_deg": cand_best_sam,
        }
    )
    print(
        f"Sweep ({cand_a},{cand_b}) | best val PSNR {cand_best_psnr:.2f} dB | "
        f"best val SAM {cand_best_sam:.2f} deg"
    )

    if (cand_best_psnr > best_candidate_psnr) or (
        abs(cand_best_psnr - best_candidate_psnr) < 1e-6 and cand_best_sam < best_candidate_sam
    ):
        best_candidate_idx = cand_idx
        best_candidate_state = cand_best_state
        best_candidate_psnr = cand_best_psnr
        best_candidate_sam = cand_best_sam

SWEEP_HISTORY_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(SWEEP_HISTORY_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(sweep_rows[0].keys()))
    writer.writeheader()
    writer.writerows(sweep_rows)

selected_idx = int(best_candidate_idx)
selected_a = int(candidate_ab[selected_idx, 0].item())
selected_b = int(candidate_ab[selected_idx, 1].item())
selected_mask = candidate_tiles[selected_idx].to(DEVICE)

model = UNet2D().to(DEVICE)
model.load_state_dict(best_candidate_state)
refine_optimizer = torch.optim.Adam(model.parameters(), lr=REFINE_LR)
refine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(refine_optimizer, T_max=FINAL_REFINE_EPOCHS)
refine_best_psnr = -float("inf")
refine_history = []

for epoch in range(1, FINAL_REFINE_EPOCHS + 1):
    model.train()
    refine_train_loss = 0.0
    refine_train_psnr = 0.0

    for x in train_loader:
        x = x.to(DEVICE)
        sensed = apply_candidate_mask(x, selected_mask)
        pred = model(sensed)
        loss = criterion(pred, x) + TRAIN_SAM_WEIGHT * spectral_angle_loss(pred, x)
        refine_optimizer.zero_grad()
        loss.backward()
        refine_optimizer.step()
        refine_train_loss += loss.item()
        refine_train_psnr += compute_psnr(pred, x).item()

    refine_scheduler.step()
    model.eval()
    refine_val_psnr = 0.0
    refine_val_sam = 0.0
    with torch.no_grad():
        for x in val_loader:
            x = x.to(DEVICE)
            pred = model(apply_candidate_mask(x, selected_mask))
            refine_val_psnr += compute_psnr(pred, x).item()
            refine_val_sam += spectral_angle_mapper(pred, x).item()

    refine_train_loss /= len(train_loader)
    refine_train_psnr /= len(train_loader)
    refine_val_psnr /= len(val_loader)
    refine_val_sam /= len(val_loader)
    refine_history.append(
        {
            "epoch": epoch,
            "train_loss": refine_train_loss,
            "train_psnr": refine_train_psnr,
            "val_psnr": refine_val_psnr,
            "val_sam_deg": refine_val_sam,
            "selected_index": selected_idx,
            "selected_a": selected_a,
            "selected_b": selected_b,
            "selected_score_raw": float(candidate_scores_raw[selected_idx].item()),
        }
    )
    print(
        f"Refine {epoch:02d} | train PSNR {refine_train_psnr:.2f} dB | "
        f"val PSNR {refine_val_psnr:.2f} dB | val SAM {refine_val_sam:.2f} deg | "
        f"fixed sel ({selected_a},{selected_b}) | lr {refine_scheduler.get_last_lr()[0]:.2e}"
    )

    if refine_val_psnr > refine_best_psnr:
        refine_best_psnr = refine_val_psnr
        REFINE_CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)
        torch.save(
            {
                "epoch": epoch,
                "model": model.state_dict(),
                "selector": selector.state_dict(),
                "best_val_psnr": refine_best_psnr,
                "selected_index": selected_idx,
                "selected_a": selected_a,
                "selected_b": selected_b,
                "selected_score_raw": float(candidate_scores_raw[selected_idx].item()),
                "geometry_optimal_indices": geometry_indices,
                "sweep_rows": sweep_rows,
                "candidate_ab": candidate_ab.cpu().numpy(),
                "candidate_scores_raw": candidate_scores_raw.cpu().numpy(),
                "candidate_probs": stage1_checkpoint["candidate_probs"],
                "hard_tile": (candidate_tiles[selected_idx].argmax(dim=0) + 1).detach().cpu().numpy(),
            },
            REFINE_CKPT_PATH,
        )

REFINE_HISTORY_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(REFINE_HISTORY_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(refine_history[0].keys()))
    writer.writeheader()
    writer.writerows(refine_history)


In [ ]:
checkpoint = torch.load(REFINE_CKPT_PATH, map_location=DEVICE, weights_only=False)
selected_idx = int(checkpoint["selected_index"])
selected_a = int(checkpoint["selected_a"])
selected_b = int(checkpoint["selected_b"])
hard_tile = checkpoint["hard_tile"]
probs = checkpoint["candidate_probs"]
candidate_ab_np = checkpoint["candidate_ab"]
candidate_scores_np = checkpoint["candidate_scores_raw"]
sweep_rows = checkpoint.get("sweep_rows", [])

best_osp_idx = int(np.argmax(candidate_scores_np))
best_osp_ab = tuple(int(v) for v in candidate_ab_np[best_osp_idx].tolist())

labels = [f"({int(a)},{int(b)})" for a, b in candidate_ab_np.tolist()]
top_order = np.argsort(probs)[::-1]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.imshow(hard_tile, cmap="tab20")
plt.title(f"Learned exact OSP tile\nselected (a,b)=({selected_a},{selected_b})")
plt.colorbar()

plt.subplot(1, 2, 2)
plt.bar(range(len(probs)), probs, color="tab:blue")
plt.xticks(range(len(probs)), labels, rotation=45, ha="right")
plt.title("Candidate selection probabilities")
plt.ylabel("Probability")
plt.tight_layout()
FIG_PATH.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG_PATH, dpi=200, bbox_inches="tight")
plt.show()

print("Exact best OSP candidate:", best_osp_ab, "score", float(candidate_scores_np[best_osp_idx]))
print("Learned selected candidate:", (selected_a, selected_b), "score", float(checkpoint["selected_score_raw"]))
print("Top-3 learned candidates:", [(labels[i], float(probs[i])) for i in top_order[:3]])
print("Best refined val PSNR:", float(checkpoint["best_val_psnr"]))
if sweep_rows:
    print("Hard-sweep summary:", sweep_rows)


Use this notebook when the final filter must remain inside the exact discrete OSP family.

It is the strictest answer to:
"Can the OSP parameter-selection step itself be made learnable?"
